In [1]:
import sys
from pathlib import Path
import pandas as pd



project_root = Path.cwd().parent
sys.path.append(str(project_root))


data_raw = Path("../data/raw")
retail = data_raw / "Online-Retail.xlsx"
print(retail)
print(retail.exists())


..\data\raw\Online-Retail.xlsx
True


# Extração do DataFrame

In [2]:
from src.extract import extract_excel

df_extraido = extract_excel(retail)

print("Dados extraídos:", df_extraido.shape)
df_extraido.head()

Dados extraídos: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df_extraido.shape

(541909, 8)

In [4]:
df_extraido.empty


False

# Transformação do DataFrame

In [5]:
from src.transform import transform_data

df_transformado = transform_data(df_extraido)

print("Dados transformados:", df_transformado.shape)

Registros antes: 541909
Duplicados encontrados: 5268
Registros depois: 536641
Quantidade de registros cancelados: 9251
Quantidade de registros de ajuste de estoque: 1336
Quantidade de vendas: 526054
Quantidade de cancelamentos: 9251
Quantidade de ajustes de estoque: 1336
Dados transformados: (536641, 12)


In [6]:
df_transformado.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'cancelado', 'ajuste_estoque',
       'tipo_transacao', 'valor_total'],
      dtype='str')

In [7]:
df_transformado.shape

(536641, 12)

In [8]:
df_transformado[
    [
        "InvoiceNo",
        "Quantity",
        "UnitPrice",
        "cancelado",
        "ajuste_estoque",
        "tipo_transacao",
        "valor_total"
    ]
].head(10)

,InvoiceNo,Quantity,UnitPrice,cancelado,ajuste_estoque,tipo_transacao,valor_total
0,536365,6,2.55,False,False,VENDA,15.30
1,536365,6,3.39,False,False,VENDA,20.34
2,536365,8,2.75,False,False,VENDA,22.00
3,536365,6,3.39,False,False,VENDA,20.34
4,536365,6,3.39,False,False,VENDA,20.34
5,536365,2,7.65,False,False,VENDA,15.30
6,536365,6,4.25,False,False,VENDA,25.50
7,536366,6,1.85,False,False,VENDA,11.10
8,536366,6,1.85,False,False,VENDA,11.10
9,536367,32,1.69,False,False,VENDA,54.08


In [9]:
df_trasnformado = transform_data(df_extraido)

print("dados extraidos", df_extraido.shape)
print("dados transformados", df_trasnformado.shape)

Registros antes: 541909
Duplicados encontrados: 5268
Registros depois: 536641
Quantidade de registros cancelados: 9251
Quantidade de registros de ajuste de estoque: 1336
Quantidade de vendas: 526054
Quantidade de cancelamentos: 9251
Quantidade de ajustes de estoque: 1336
dados extraidos (541909, 8)
dados transformados (536641, 12)


### Testes de validação do schema

Nesta etapa, testamos a identificação de colunas obrigatórias ausentes no DataFrame.

Para simular esse cenário, removemos intencionalmente a coluna `Quantity` e verificamos se a função `extract_excel()` identificava o problema.

**Resultado:** a função lançou um `ValueError` informando a coluna ausente, impedindo que dados com estrutura incompleta fossem enviados para a etapa de transformação.

In [10]:


df_teste = pd.DataFrame(columns=["Produto", "Preco"])

print(df_teste.empty)
print(df_teste.shape)

True
(0, 2)


In [11]:
colunas_esperadas = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
}

colunas_encontradas = set(df_extraido.columns)

print(colunas_esperadas - colunas_encontradas)

set()


In [12]:
df_teste_schema = df_extraido.head(5).drop(columns=["Quantity"])

colunas_encontradas = set(df_teste_schema.columns)

colunas_faltantes = colunas_esperadas - colunas_encontradas

print(colunas_faltantes)

{'Quantity'}


In [17]:
arquivo_schema = "../data/raw/teste_schema.xlsx"
try:
    extract_excel(arquivo_schema)

except ValueError as erro:
    print("Teste aprovado!")
    print("Erro identificado:", erro)
else:
    print("Teste falhou: a função aceitou um arquivo inválido.")

Teste aprovado!
Erro identificado: Colunas faltantes no DataFrame extraído: {'Quantity'}
